# Phase 5 — Demand Forecasting (Time Series with Prophet)

**Goal:** Predict future monthly order volumes using historical time series data.

**Why this matters in interviews:**
E-commerce companies use demand forecasting for:
- **Inventory planning** — how much stock to order from suppliers
- **Logistics capacity** — how many delivery staff/trucks needed
- **Revenue projections** — financial planning
- **Marketing budget allocation** — spend more during high-demand months

**What is a Time Series?**
A sequence of data points indexed in time order. Unlike regular regression where rows are independent, time series data has **temporal dependencies** — today's value depends on past values (autocorrelation).

**Model: Facebook Prophet**
Prophet (open-sourced by Meta in 2017) decomposes a time series into:
- **Trend**: long-term growth or decline
- **Seasonality**: repeating patterns (weekly, monthly, yearly)
- **Holidays**: special events that cause spikes/dips
- **Residuals**: noise

It's robust to missing data and outliers — ideal for messy real-world business data.

## 0. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (14, 5), 'axes.titlesize': 14})

# Connect to Supabase
load_dotenv()
engine = create_engine(os.getenv('SUPABASE_URL'))

# Load orders and payments
query = """
    SELECT 
        o.order_id,
        o.order_purchase_timestamp,
        o.order_status,
        COALESCE(SUM(p.payment_value), 0) AS revenue
    FROM orders o
    LEFT JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id, o.order_purchase_timestamp, o.order_status
"""
df_raw = pd.read_sql(query, engine)
print(f'Loaded {len(df_raw):,} delivered orders')
df_raw.head()

## 1. Build the Monthly Time Series

We aggregate individual orders into **monthly buckets**. This is called **resampling** — converting a high-frequency signal (daily) to a lower frequency (monthly).

**Why monthly?** The Olist dataset covers about 2 years (Sep 2016 to Oct 2018). Monthly aggregation gives us ~25 data points — enough for trend analysis but too few for daily forecasting.

`pd.Grouper(freq='MS')` groups by **Month Start** — the first day of each month.

In [ ]:
# Parse datetime
df_raw['order_date'] = pd.to_datetime(df_raw['order_purchase_timestamp'], errors='coerce')
df_raw = df_raw.dropna(subset=['order_date'])

# Monthly aggregation: order count + revenue
df_monthly = (
    df_raw
    .set_index('order_date')
    .resample('MS')   # Month Start frequency
    .agg(
        order_count=('order_id', 'count'),
        monthly_revenue=('revenue', 'sum')
    )
    .reset_index()
    .rename(columns={'order_date': 'month'})
)

# Remove first and last months (often incomplete)
df_monthly = df_monthly.iloc[1:-1].reset_index(drop=True)

print(f'Time range: {df_monthly["month"].min().strftime("%b %Y")} to {df_monthly["month"].max().strftime("%b %Y")}')
print(f'Total months: {len(df_monthly)}')
print(f'Total orders in period: {df_monthly["order_count"].sum():,}')
print(f'Total revenue in period: BRL {df_monthly["monthly_revenue"].sum():,.2f}')
df_monthly.tail(10)

## 2. Visualize the Raw Time Series

Before modeling, always plot the raw data. Look for:
- **Trend**: is it going up, down, or flat?
- **Seasonality**: are there repeating peaks/troughs?
- **Outliers/anomalies**: sudden spikes or drops?

Olist's growth story is visible in the data: rapid growth from late 2016 to late 2017, then plateau.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Order count trend
axes[0].plot(df_monthly['month'], df_monthly['order_count'], 
             color='steelblue', linewidth=2.5, marker='o', markersize=5)
axes[0].fill_between(df_monthly['month'], df_monthly['order_count'], alpha=0.15, color='steelblue')
axes[0].set_title('Monthly Order Volume — Olist (2016-2018)', fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Revenue trend
axes[1].plot(df_monthly['month'], df_monthly['monthly_revenue'] / 1000, 
             color='darkorange', linewidth=2.5, marker='s', markersize=5)
axes[1].fill_between(df_monthly['month'], df_monthly['monthly_revenue'] / 1000, alpha=0.15, color='darkorange')
axes[1].set_title('Monthly Revenue (BRL 000s)', fontweight='bold')
axes[1].set_ylabel('Revenue (BRL, thousands)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')

# Annotate peak
peak_idx = df_monthly['order_count'].idxmax()
peak_month = df_monthly.loc[peak_idx, 'month']
peak_val = df_monthly.loc[peak_idx, 'order_count']
axes[0].annotate(f'Peak: {peak_val:,} orders\n{peak_month.strftime("%b %Y")}',
                 xy=(peak_month, peak_val), xytext=(peak_month, peak_val * 0.75),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9, ha='center')

plt.tight_layout()
plt.show()

# Growth metrics
first_full = df_monthly.iloc[0]
last_full = df_monthly.iloc[-1]
growth = (last_full['order_count'] - first_full['order_count']) / first_full['order_count'] * 100
print(f'Order growth from {first_full["month"].strftime("%b %Y")} to {last_full["month"].strftime("%b %Y")}: {growth:.1f}%')

## 3. Train/Test Split

**Critical rule for time series:** You CANNOT use random train/test split (like in regular ML). Time series data has a **temporal order** — you can only predict the future using past data.

If you randomly split, some training data would be from the future relative to some test data — this is called **data leakage** and would give falsely optimistic results.

**We use the last 3 months as test set** (hold-out period) and train on everything before.

In [ ]:
TEST_MONTHS = 3

df_train = df_monthly.iloc[:-TEST_MONTHS].copy()
df_test  = df_monthly.iloc[-TEST_MONTHS:].copy()

print(f'Training set: {len(df_train)} months ({df_train["month"].min().strftime("%b %Y")} to {df_train["month"].max().strftime("%b %Y")})')
print(f'Test set:     {len(df_test)} months ({df_test["month"].min().strftime("%b %Y")} to {df_test["month"].max().strftime("%b %Y")})')

# Visualize the split
plt.figure(figsize=(14, 5))
plt.plot(df_train['month'], df_train['order_count'], 'b-o', label='Training data', linewidth=2)
plt.plot(df_test['month'],  df_test['order_count'],  'ro-', label='Test data (held out)', linewidth=2)
plt.axvline(x=df_test['month'].iloc[0], color='gray', linestyle='--', label='Train/Test split')
plt.title('Train / Test Split — Time-Based (No Random Shuffling!)', fontweight='bold')
plt.ylabel('Monthly Orders')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Baseline Model — Naive Forecast

Before building a complex ML model, always establish a **baseline**. The simplest possible forecast is the **naive/persistence model**: predict next month = last month's value.

If your ML model can't beat this naive baseline, it's not adding value.

**Metrics:**
- **MAE** (Mean Absolute Error): average absolute difference between predicted and actual. In same units as the target.
- **RMSE** (Root Mean Squared Error): penalizes large errors more than MAE.
- **MAPE** (Mean Absolute Percentage Error): percentage error — easier to communicate to stakeholders ("we were off by 8% on average").

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mape(actual, predicted):
    """Mean Absolute Percentage Error — ignores months with zero actual."""
    actual, predicted = np.array(actual), np.array(predicted)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# Naive model: last value from training set, repeated for all test months
last_train_value = df_train['order_count'].iloc[-1]
naive_pred = [last_train_value] * len(df_test)

actual_test = df_test['order_count'].values

naive_mae  = mean_absolute_error(actual_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(actual_test, naive_pred))
naive_mape = mape(actual_test, naive_pred)

print('BASELINE (Naive / Persistence) Model:')
print(f'  MAE:  {naive_mae:.0f} orders')
print(f'  RMSE: {naive_rmse:.0f} orders')
print(f'  MAPE: {naive_mape:.1f}%')

## 5. Prophet Model

**Prophet requires a specific DataFrame format:**
- Column `ds`: date column (datetime)
- Column `y`: the target variable (order count)

Prophet fits a **generalized additive model**: `y(t) = trend(t) + seasonality(t) + holidays(t) + ε`

After fitting, `predict()` returns a DataFrame with `yhat` (forecast), `yhat_lower`, `yhat_upper` (uncertainty intervals).

**Install if needed:** `pip install prophet`

In [ ]:
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    print('Prophet not installed. Run: pip install prophet')
    print('Falling back to a simple linear trend baseline...')
    PROPHET_AVAILABLE = False

if PROPHET_AVAILABLE:
    # Format data for Prophet
    prophet_train = df_train.rename(columns={'month': 'ds', 'order_count': 'y'})[['ds', 'y']]
    prophet_test  = df_test.rename(columns={'month': 'ds', 'order_count': 'y'})[['ds', 'y']]

    # Initialize and fit Prophet
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,  # monthly data — no weekly pattern
        daily_seasonality=False,
        seasonality_mode='multiplicative',  # seasonality scales with trend
        changepoint_prior_scale=0.1,        # controls trend flexibility (higher = more flexible)
        interval_width=0.95                 # 95% confidence intervals
    )
    model.fit(prophet_train)
    print('Prophet model trained successfully!')

    # Create future dataframe for prediction (covers test period)
    future = model.make_future_dataframe(periods=TEST_MONTHS + 3, freq='MS')
    forecast = model.predict(future)
    
    # Extract test period predictions
    test_forecast = forecast[forecast['ds'].isin(df_test['month'])]
    prophet_pred = test_forecast['yhat'].values
    prophet_pred = np.maximum(prophet_pred, 0)  # no negative orders

    prophet_mae  = mean_absolute_error(actual_test, prophet_pred)
    prophet_rmse = np.sqrt(mean_squared_error(actual_test, prophet_pred))
    prophet_mape = mape(actual_test, prophet_pred)

    print('\nProphet Model Results:')
    print(f'  MAE:  {prophet_mae:.0f} orders')
    print(f'  RMSE: {prophet_rmse:.0f} orders')
    print(f'  MAPE: {prophet_mape:.1f}%')
    
    improvement = (naive_mae - prophet_mae) / naive_mae * 100
    print(f'\nProphet vs Naive: {improvement:+.1f}% improvement in MAE')

## 6. Visualize Forecast vs Actuals

The forecast plot is what you'll show in your portfolio and in interviews. It must clearly show:
1. Historical training data
2. The model's forecast with uncertainty bands
3. Actual test values overlaid (so we can see how well it predicted)

In [ ]:
if PROPHET_AVAILABLE:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # --- Plot 1: Full forecast ---
    ax = axes[0]
    
    # Training actuals
    ax.plot(df_train['month'], df_train['order_count'], 'b-o', 
            label='Training Actuals', linewidth=2, markersize=5)
    
    # Test actuals
    ax.plot(df_test['month'], df_test['order_count'], 'go', 
            label='Test Actuals', markersize=10, zorder=5)
    
    # Forecast line
    ax.plot(forecast['ds'], forecast['yhat'], 'r--', label='Prophet Forecast', linewidth=2)
    
    # Confidence intervals
    ax.fill_between(forecast['ds'], forecast['yhat_lower'], forecast['yhat_upper'],
                   alpha=0.15, color='red', label='95% Confidence Interval')
    
    # Train/test split line
    ax.axvline(x=df_test['month'].iloc[0], color='gray', linestyle=':', linewidth=1.5)
    ax.text(df_test['month'].iloc[0], ax.get_ylim()[1] * 0.9, ' Test Period', color='gray', fontsize=9)
    
    ax.set_title('Prophet Demand Forecast — Monthly Orders', fontweight='bold')
    ax.set_ylabel('Monthly Orders')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

    # --- Plot 2: Forecast components ---
    ax2 = axes[1]
    ax2.bar(df_test['month'], actual_test, label='Actual', color='steelblue', alpha=0.7, width=20)
    ax2.bar(df_test['month'], prophet_pred, label='Predicted', color='tomato', alpha=0.7, width=20, 
            bottom=0)
    for i, (a, p) in enumerate(zip(actual_test, prophet_pred)):
        err = abs(a - p)
        ax2.text(df_test['month'].iloc[i], max(a, p) + 50, f'Error: {err:.0f}', 
                ha='center', fontsize=9, color='darkred')
    ax2.set_title('Test Period: Actual vs Predicted Orders', fontweight='bold')
    ax2.set_ylabel('Orders')
    ax2.legend()
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    plt.suptitle(f'Prophet Forecast | MAE: {prophet_mae:.0f} orders | MAPE: {prophet_mape:.1f}%', 
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 7. Prophet Components — What Drives the Forecast?

In [ ]:
if PROPHET_AVAILABLE:
    # Prophet's built-in component plot decomposes the forecast
    fig = model.plot_components(forecast)
    fig.suptitle('Prophet Decomposition: Trend + Yearly Seasonality', fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('The trend component shows Olist\'s growth curve.')
    print('The yearly seasonality shows which months typically have higher/lower order volumes.')

## 8. Metrics Summary & Model Comparison

In [ ]:
print('=' * 50)
print('MODEL COMPARISON SUMMARY')
print('=' * 50)

results = [
    {'Model': 'Naive (Baseline)',  'MAE': naive_mae,  'RMSE': naive_rmse,  'MAPE (%)': naive_mape}
]
if PROPHET_AVAILABLE:
    results.append({'Model': 'Facebook Prophet', 'MAE': prophet_mae, 'RMSE': prophet_rmse, 'MAPE (%)': prophet_mape})

df_results = pd.DataFrame(results).set_index('Model')
print(df_results.round(2).to_string())

print('\n--- Interpretation ---')
print('MAE  = average absolute error in orders (easy to explain to business)')
print('RMSE = penalizes large errors more (sensitive to outlier months)')
print('MAPE = percentage error (best for communicating accuracy)')

## 9. Interview Talking Points

1. **Why time-based train/test split?** Time series data has temporal dependencies. Randomly splitting would leak future data into training (data leakage), giving unrealistically optimistic results. Always split chronologically.

2. **What is MAPE and when to use it?** Mean Absolute Percentage Error expresses error as a percentage of the actual value — scale-independent and easy to explain to non-technical stakeholders. Watch out: it's undefined when actual = 0 and penalizes under-prediction more than over-prediction.

3. **What is Prophet's trend model?** Prophet uses a piecewise linear or logistic growth trend with **changepoints** — points in time where the trend slope changes. The `changepoint_prior_scale` hyperparameter controls how flexible the trend is.

4. **What are the limitations of this forecast?** Only ~25 monthly data points limits statistical power. The dataset ends in 2018 — no data on pandemic effects or later platform changes. External regressors (marketing spend, promotions) are not included but could significantly improve accuracy.

5. **How would you improve this in production?** Add external regressors (holiday calendar, promotional events), use more granular data (daily/weekly), retrain on a rolling window, deploy with automated retraining, monitor forecast drift.